# `nue` - Feature Contribution Analysis

`megumi.nue` answers a single question: **"If I add these features, how much improvement do I get, and is it statistically significant?"**

Under the hood it fits two vanilla models per cross-validation fold, one with the base features alone, one with base + candidate features. Then runs a paired t-test across fold scores to assess whether the difference is beyond noise.

This notebook covers four scenarios across two task types:

| # | Task | New features | Expected result |
|---|---|---|---|
| 1.1 | Classification | Genuine signal | Significant positive delta |
| 1.2 | Classification | Pure noise | No significant change |
| 1.3 | Classification | Genuine signal + UDF metric | Business value in currency |
| 2.1 | Regression | Genuine signal | Significant negative delta on RMSE |
| 2.2 | Regression | Pure noise | No significant change |

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression

from megumi.nue import evaluate_contribution

rng = np.random.default_rng(42)

---
## 1. Binary classification

**Scenario:** A lending company has a credit model trained on five internal features (`base_0` … `base_4`). A data vendor offers five additional scores. We want to know whether they are worth buying.

We build a dataset with **10 independently informative features** (no redundancy, no repeated features). The first five go to the base model; the second five are the candidate features. This is to make an example of a case when the vendor features are strong predictors and when they are not.

In [2]:
N = 4000
base_clf = [f"base_{i}" for i in range(5)]
vendor_clf = [f"vendor_{i}" for i in range(5)]
noise_clf = [f"noise_{i}" for i in range(5)]

X, y = make_classification(
    n_samples=N,
    n_features=10,
    n_informative=10,
    n_redundant=0,
    n_repeated=0,
    random_state=42,
)

df_clf = pd.DataFrame(X, columns=base_clf + vendor_clf)
df_clf["default"] = y
df_clf["loan_amount"] = rng.uniform(5_000, 50_000, N)

for i in range(5):
    df_clf[f"noise_{i}"] = rng.standard_normal(N)

df_clf.head(3)

,base_0,base_1,base_2,base_3,base_4,vendor_0,vendor_1,vendor_2,vendor_3,vendor_4,default,loan_amount,noise_0,noise_1,noise_2,noise_3,noise_4
0,-0.176854,0.536819,0.976507,2.527593,2.572763,-3.730282,2.627653,2.387550,-1.574774,-2.135414,1,39828.022185,-1.040601,-1.128436,-0.240521,1.091987,2.677653
1,-0.449937,-0.811552,-3.014615,1.812834,2.148779,-0.593486,0.542769,1.124727,-0.388622,-1.626846,0,24749.529789,-1.496660,-2.536430,-0.256294,1.226838,1.070822
2,3.086504,-0.837319,1.826980,-1.100126,1.342087,-1.108440,0.209519,1.731675,1.585938,-1.747063,0,43636.906396,1.352594,0.595387,-1.592317,1.287673,0.515883


### 1.1 New features with genuine signal

The vendor features are independently informative — they contain predictive signal that the base model cannot see. We expect a clear and statistically significant improvement.

In [3]:
result_good_clf = evaluate_contribution(
    df_clf,
    base_features=base_clf,
    new_features=vendor_clf,
    target="default",
    metrics=["roc_auc", "recall", "precision", "f1"],
    n_splits=10,
    random_state=42,
)

result_good_clf

,metric,base_score,augmented_score,delta,pct_change,p_value,significant
0,roc_auc,0.879252,0.988324,0.109072,12.405031,1.809159e-10,True
1,recall,0.807580,0.968520,0.160940,19.928723,2.002744e-07,True
2,precision,0.790725,0.942731,0.152005,19.223559,6.206410e-10,True
3,f1,0.798611,0.955403,0.156792,19.633045,6.457673e-09,True


All metrics show a **positive delta** and `significant=True`: the vendor features genuinely improve the model. The `pct_change` column quantifies the gain as a percentage of the base score.

### 1.2 New features that are pure noise

The noise features are drawn from a standard normal distribution — they have no relationship with the target. We expect near-zero delta and `significant=False`.

In [4]:
result_noise_clf = evaluate_contribution(
    df_clf,
    base_features=base_clf,
    new_features=noise_clf,
    target="default",
    metrics=["roc_auc", "recall", "precision", "f1"],
    n_splits=10,
    random_state=42,
)

result_noise_clf

,metric,base_score,augmented_score,delta,pct_change,p_value,significant
0,roc_auc,0.879252,0.867101,-0.012151,-1.382008,0.713757,False
1,recall,0.807580,0.807090,-0.000490,-0.060681,0.666150,False
2,precision,0.790725,0.776008,-0.014717,-1.861211,0.837174,False
3,f1,0.798611,0.790788,-0.007823,-0.979546,0.944712,False


Delta is close to zero and `significant=False` across the board: adding noise features does not meaningfully change the model. Any small delta is attributable to random forest variance across folds, not real signal.

### 1.3 Business metric: expected loss avoided (UDF)

Classification metrics tell us about model quality; the business cares about money. We define a custom metric that measures **total loan amount of true defaults that the model correctly flags** (and would presumably decline). A larger value means more potential loss avoided.

The UDF receives `(y_true, y_pred_proba, df_fold)` — the third argument is the test-fold slice of `df` with all columns, giving access to `loan_amount`.

In [5]:
def loss_avoided(
    y_true: np.ndarray,
    y_pred_proba: np.ndarray,
    df_fold: pd.DataFrame,
    threshold: float = 0.5,
):
    """Total loan amount of true defaults correctly flagged at the given threshold."""
    flagged = y_pred_proba >= threshold
    true_defaults = y_true.astype(bool)
    return df_fold.loc[true_defaults & flagged, "loan_amount"].sum()


result_biz = evaluate_contribution(
    df_clf,
    base_features=base_clf,
    new_features=vendor_clf,
    target="default",
    metrics=["roc_auc", loss_avoided],
    n_splits=10,
    random_state=42,
)

result_biz

,metric,base_score,augmented_score,delta,pct_change,p_value,significant
0,roc_auc,8.792522e-01,9.883237e-01,0.109072,12.405031,1.809159e-10,True
1,loss_avoided,4.433589e+06,5.265190e+06,831600.896200,18.756834,6.245724e-07,True


The `delta` for `loss_avoided` is the average **additional loss caught per fold** when the vendor features are included. Multiply by the number of expected folds in production and compare against the vendor's licensing cost to get a direct ROI estimate.

---
## 2. Regression

**Scenario:** A valuation model predicts property prices using five engineered features. A data provider offers five additional location-based scores. We want to know whether they reduce prediction error.

Same strategy as classification: we generate 10 independently informative features and split them 5 + 5. For the noise case we add five random columns.

For regression, **a negative delta on RMSE/MAE means improvement** (lower error). For R², a positive delta means improvement.

In [6]:
base_reg = [f"base_{i}" for i in range(5)]
provider_reg = [f"provider_{i}" for i in range(5)]
noise_reg = [f"noise_{i}" for i in range(5)]

X_reg, y_reg = make_regression(
    n_samples=N,
    n_features=10,
    n_informative=10,
    noise=25,
    random_state=42,
)

df_reg = pd.DataFrame(X_reg, columns=base_reg + provider_reg)
df_reg["price"] = y_reg

for i in range(5):
    df_reg[f"noise_{i}"] = rng.standard_normal(N)

df_reg.head(3)

,base_0,base_1,base_2,base_3,base_4,provider_0,provider_1,provider_2,provider_3,provider_4,price,noise_0,noise_1,noise_2,noise_3,noise_4
0,-1.793921,-0.386542,-0.566705,-0.697719,-0.334803,-0.653463,1.228030,1.243406,-0.599821,-0.657135,-59.950648,0.258215,-0.964251,-2.096192,0.060080,1.301741
1,0.092421,0.021665,0.135283,-0.241315,1.341330,-0.829143,-0.181645,-0.473523,-1.447218,-0.309244,-119.614667,0.318527,0.773045,0.208731,-0.398823,0.414907
2,-1.684072,-0.121359,0.889260,-0.837730,1.129920,0.566322,-0.541243,-0.215956,-1.744258,0.537216,-113.776834,-0.557474,-1.180878,-0.530565,-2.863277,0.518008


### 2.1 New features with genuine signal

In [7]:
result_good_reg = evaluate_contribution(
    df_reg,
    base_features=base_reg,
    new_features=provider_reg,
    target="price",
    metrics=["rmse", "mae", "r2"],
    n_splits=10,
    random_state=42,
)

result_good_reg

,metric,base_score,augmented_score,delta,pct_change,p_value,significant
0,rmse,154.930842,68.692852,-86.237990,-55.662248,8.782379e-13,True
1,mae,123.466930,52.822963,-70.643967,-57.216914,4.018462e-12,True
2,r2,0.211588,0.844965,0.633377,299.344497,2.348318e-12,True


RMSE and MAE show a **negative delta** (lower error) and R² shows a **positive delta** (more variance explained), all marked `significant=True`.

### 2.2 New features that are pure noise

In [8]:
result_noise_reg = evaluate_contribution(
    df_reg,
    base_features=base_reg,
    new_features=noise_reg,
    target="price",
    metrics=["rmse", "mae", "r2"],
    n_splits=10,
    random_state=42,
)

result_noise_reg

,metric,base_score,augmented_score,delta,pct_change,p_value,significant
0,rmse,154.930842,153.039346,-1.891495,-1.220864,0.196406,False
1,mae,123.466930,122.278422,-1.188508,-0.962613,0.646461,False
2,r2,0.211588,0.230761,0.019173,9.061590,0.208116,False


Delta is negligible and `significant=False`: the noise features add nothing to the model's ability to predict price.